hier werden bestehende 1 min Daten zusätzlich in 5 um 30 min Daten umgewandelt und gespeichert 

In [ ]:
import pandas as pd
from datetime import timedelta
import os
import sys
from sqlalchemy import func
from sqlalchemy.dialects.mysql import insert  # Import für MySQL-Dialekt

current_dir = os.getcwd()
sys.path.append(os.path.abspath(os.path.join(current_dir, '..')))
import modules.SQLAlchemy_functions as af

# Funktion, um Daten einzufügen und Duplikate zu aktualisieren
def insert_data_with_update(session, data, table_class, symbol_id):
    for _, row in data.iterrows():
        # Erstelle das Insert-Statement
        insert_stmt = insert(table_class).values(
            date=row['date'],
            open=row['open'],
            high=row['high'],
            low=row['low'],
            close=row['close'],
            volume=row['volume'],
            symbol_id=symbol_id
        )
        # Füge ON DUPLICATE KEY UPDATE hinzu, um bestehende Einträge zu aktualisieren
        update_stmt = insert_stmt.on_duplicate_key_update(
            open=row['open'],
            high=row['high'],
            low=row['low'],
            close=row['close'],
            volume=row['volume']
        )
        session.execute(update_stmt)
    session.commit()

# Funktion, um Daten in Blöcken zu laden und zu resampeln
def resample_data_in_batches(session, symbol_id, interval, table_class, chunk_size=10000):
    # Bestimme das früheste und späteste Datum für das Symbol in der MinuteBar-Tabelle
    min_date = session.query(func.min(af.MinuteBar.date)).filter(af.MinuteBar.symbol_id == symbol_id).scalar()
    max_date = session.query(func.max(af.MinuteBar.date)).filter(af.MinuteBar.symbol_id == symbol_id).scalar()

    # Überprüfe, ob min_date und max_date gültig sind
    if min_date is None or max_date is None:
        print(f"Keine Daten für Symbol-ID {symbol_id} in der MinuteBar-Tabelle gefunden.")
        return  # Beende die Funktion, wenn keine Daten vorhanden sind

    current_date = min_date
    
    while current_date < max_date:
        # Definiere das nächste Zeitfenster
        next_date = current_date + timedelta(minutes=chunk_size)
        
        # Lade den aktuellen Chunk der Daten aus der MinuteBar-Tabelle
        chunk_data = pd.read_sql(
            session.query(af.MinuteBar)
                   .filter(
                       af.MinuteBar.symbol_id == symbol_id,
                       af.MinuteBar.date >= current_date,
                       af.MinuteBar.date < next_date
                   ).statement,
            session.bind
        )
        
        # Prüfe, ob Daten im Chunk vorhanden sind
        if chunk_data.empty:
            current_date = next_date
            continue
        
        # Setze 'date' als Index und konvertiere zu datetime für Resampling
        chunk_data['date'] = pd.to_datetime(chunk_data['date'])
        chunk_data.set_index('date', inplace=True)

        # Resample den Chunk
        resampled_chunk = chunk_data.resample(interval).agg({
            'open': 'first',
            'high': 'max',
            'low': 'min',
            'close': 'last',
            'volume': 'sum'
        }).dropna().reset_index()

        # Füge die resampelten Daten in die Ziel-Tabelle ein
        insert_data_with_update(session, resampled_chunk, table_class, symbol_id)

        # Aktualisiere das Zeitfenster
        current_date = next_date

    print(f"{interval}-Resampling für Symbol-ID {symbol_id} abgeschlossen.")

# Hauptfunktion für das Batch-Resampling
def main_resampling_with_batches(session, symbol_ids, chunk_size=10000):
    for symbol_id in symbol_ids:
        print(f"Starte Batch-Resampling für Symbol-ID {symbol_id}...")
        
        # Resampling für 5 Minuten
        resample_data_in_batches(session, symbol_id, '5min', af.FiveMinuteBar, chunk_size)
        
        # Resampling für 30 Minuten
        resample_data_in_batches(session, symbol_id, '30min', af.ThirtyMinuteBar, chunk_size)


#optional
def insert_data_on_duplicate_key_update(session, data, table_class):
    insert_stmt = insert(table_class).values(data)
    update_stmt = insert_stmt.on_duplicate_key_update({
        col.name: col for col in insert_stmt.inserted
    })
    session.execute(update_stmt)
    session.commit()

In [2]:
# Starte das Batch-Resampling-Skript
if __name__ == "__main__":
    config_path = 'config.json'
    session = af.start_session(config_path)

    if session is not None:
        symbol_ids = [symbol.id for symbol in session.query(af.Symbol).all()]
        
        main_resampling_with_batches(session, symbol_ids, chunk_size=10000)

        session.close()


pymysql ist bereits installiert.
cryptography ist bereits installiert.
Session und Tabellen für Datenbank erfolgreich erstellt.
Starte Batch-Resampling für Symbol-ID 1...
Keine Daten für Symbol-ID 1 in der MinuteBar-Tabelle gefunden.
Keine Daten für Symbol-ID 1 in der MinuteBar-Tabelle gefunden.
Starte Batch-Resampling für Symbol-ID 2...
Keine Daten für Symbol-ID 2 in der MinuteBar-Tabelle gefunden.
Keine Daten für Symbol-ID 2 in der MinuteBar-Tabelle gefunden.
Starte Batch-Resampling für Symbol-ID 3...
Keine Daten für Symbol-ID 3 in der MinuteBar-Tabelle gefunden.
Keine Daten für Symbol-ID 3 in der MinuteBar-Tabelle gefunden.
Starte Batch-Resampling für Symbol-ID 4...
Keine Daten für Symbol-ID 4 in der MinuteBar-Tabelle gefunden.
Keine Daten für Symbol-ID 4 in der MinuteBar-Tabelle gefunden.
Starte Batch-Resampling für Symbol-ID 5...
Keine Daten für Symbol-ID 5 in der MinuteBar-Tabelle gefunden.
Keine Daten für Symbol-ID 5 in der MinuteBar-Tabelle gefunden.
Starte Batch-Resampling für

/tmp/ipykernel_3882590/1734195886.py:49: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  resampled_chunk = chunk_data.resample(interval).agg({
/tmp/ipykernel_3882590/1734195886.py:49: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  resampled_chunk = chunk_data.resample(interval).agg({


IntegrityError: (pymysql.err.IntegrityError) (1062, "Duplicate entry '231-2023-06-27 15:05:00' for key 'five_minute_bar.symbol_id'")
[SQL: INSERT INTO five_minute_bar (date, open, high, low, close, volume, symbol_id) VALUES (%(date)s, %(open)s, %(high)s, %(low)s, %(close)s, %(volume)s, %(symbol_id)s)]
[parameters: {'date': Timestamp('2023-06-27 15:05:00'), 'open': 0.2837, 'high': 0.2837, 'low': 0.28338, 'close': 0.28338, 'volume': 2631.29, 'symbol_id': 231}]
(Background on this error at: https://sqlalche.me/e/20/gkpj)